# Task 14 — Stieltjes/log-mixture preregistered gate

Fresh public clone at an immutable raw SHA. CPU only; no Lean and no credentials.

In [ ]:
import datetime, hashlib, json, os, platform, shutil, subprocess, tempfile, time
from pathlib import Path

EXPECTED_SHA = '3fbb42e2ee73ddd37d15c45eba31e1ce7462d5ab'
EXPECTED_GATE_SHA256 = '4375cc616e6e2dee7844be31900c8558542c0891dcf976b3c3fb0d41eae56c8e'
REPO_URL = 'https://github.com/lluiseriksson/THE-ERIKSSON-PROGRAMME.git'
WORK = Path(tempfile.mkdtemp(prefix='spatial-stieltjes-gate-'))
REPO = WORK / 'repo'
ARTIFACTS = WORK / 'artifacts'
ARTIFACTS.mkdir()
transcript = []
utc_start = datetime.datetime.now(datetime.timezone.utc)

def log(message):
    text = str(message)
    transcript.append(text)
    print(text, flush=True)

def run(cmd, *, cwd=None):
    shown = ' '.join(map(str, cmd))
    log(f'$ {shown}')
    start = time.perf_counter()
    process = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    elapsed = time.perf_counter() - start
    log(process.stdout.rstrip())
    log(f'[exit {process.returncode}; elapsed {elapsed:.6f} s]')
    if process.returncode:
        (ARTIFACTS / 'failure_transcript.txt').write_text('\n'.join(transcript) + '\n', encoding='utf-8')
        raise RuntimeError(f'command failed: {shown}')
    return process, elapsed

log(f'utc_start={utc_start.isoformat()}')
log(f'runtime={platform.platform()}')
log(f'python={platform.python_version()}')
log(f'cpu_count={os.cpu_count()}')
log(Path('/proc/cpuinfo').read_text(errors='replace').split('model name', 1)[1].splitlines()[0].lstrip('\t: '))
log(Path('/proc/meminfo').read_text(errors='replace').splitlines()[0])
log('gpu=none (CPU runtime requested)')
run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO)])
run(['git', 'checkout', '--detach', EXPECTED_SHA], cwd=REPO)
head = run(['git', 'rev-parse', 'HEAD'], cwd=REPO)[0].stdout.strip()
if head != EXPECTED_SHA:
    raise RuntimeError(f'HEAD mismatch: {head}')
gate = REPO / 'scripts' / 'judge_spatial_stieltjes_log_mixture.py'
gate_hash = hashlib.sha256(gate.read_bytes()).hexdigest()
log(f'gate_sha256={gate_hash}')
if gate_hash != EXPECTED_GATE_SHA256:
    raise RuntimeError(f'gate hash mismatch: {gate_hash}')

normal, normal_seconds = run(['/usr/bin/python3', str(gate)])
optimized, optimized_seconds = run(['/usr/bin/python3', '-O', str(gate)])
if normal.stdout != optimized.stdout:
    raise RuntimeError('normal and optimized outputs differ')
payload = json.loads(normal.stdout)
expected = {
    'status': 'PASS', 'cells': 45, 'endpoint_controls': 9,
    'half_mutations_rejected': 45, 'base_mutations_rejected': 45,
    'scale_mutations_rejected': 45,
    'generic_hypotheses': ['1 < c', '0 < B', '0 <= s'],
    'physical_hypotheses': ['0 < beta', '0 < gamma', 'gamma < a'],
}
for key, value in expected.items():
    if payload.get(key) != value:
        raise RuntimeError(f'payload mismatch for {key}: {payload.get(key)!r}')
output_hash = hashlib.sha256(normal.stdout.encode()).hexdigest()
(ARTIFACTS / 'normal.json').write_text(normal.stdout, encoding='utf-8')
(ARTIFACTS / 'optimized.json').write_text(optimized.stdout, encoding='utf-8')
metadata = {
    'repo_sha': head, 'gate_sha256': gate_hash, 'output_sha256': output_hash,
    'normal_seconds': normal_seconds, 'optimized_seconds': optimized_seconds,
    'utc_start': utc_start.isoformat(),
    'utc_end': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'runtime': platform.platform(), 'python': platform.python_version(),
    'cpu_count': os.cpu_count(),
    'memory': Path('/proc/meminfo').read_text(errors='replace').splitlines()[0],
    'gpu': 'none',
}
(ARTIFACTS / 'metadata.json').write_text(json.dumps(metadata, indent=2, sort_keys=True) + '\n', encoding='utf-8')
log(f'output_sha256={output_hash}')
log(f'normal_seconds={normal_seconds:.6f}')
log(f'optimized_seconds={optimized_seconds:.6f}')
log('SPATIAL STIELTJES LOG-MIXTURE GATE PASS')
(ARTIFACTS / 'transcript.txt').write_text('\n'.join(transcript) + '\n', encoding='utf-8')
hashes = {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in sorted(ARTIFACTS.iterdir())}
(ARTIFACTS / 'SHA256SUMS').write_text(''.join(f'{digest}  {name}\n' for name, digest in hashes.items()), encoding='utf-8')
archive = shutil.make_archive('/content/spatial_stieltjes_gate_artifacts', 'zip', ARTIFACTS)
archive_hash = hashlib.sha256(Path(archive).read_bytes()).hexdigest()
print(f'artifact_zip={archive}', flush=True)
print(f'artifact_zip_sha256={archive_hash}', flush=True)
from google.colab import files
files.download(archive)
